# Demo MDLM

In [3]:
import os, sys, math, time, json
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10
from torchvision.transforms import v2
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

print(f"device   : {device}", torch.cuda.get_device_name(0) if device.type == "cuda" else "")
print(f"cpu count: {os.cpu_count()}")

device   : cuda Tesla T4
cpu count: 72


In [4]:
@torch.no_grad()
def mdlm_step(model, x, t, s, mask_id):
    B, L = x.shape
    sigma = torch.zeros(B, device=x.device)
    logits = model(x, sigma)
    logits = logits[..., :mask_id]
    probs = logits.softmax(-1)
    V = probs.shape[-1]
    token = torch.multinomial(probs.view(B * L, V), num_samples=1).view(B, L)

    u = torch.rand(x.shape, device=x.device)
    est_masque = (x == mask_id)
    reste_masque = u < s / t

    interieur = torch.where(reste_masque, mask_id, token)
    x_next = torch.where(est_masque, interieur, x)
    return x_next


def mdlm_sample(model, B, L, T, mask_id):
    x = torch.full((B, L), mask_id, dtype=torch.long, device=next(model.parameters()).device)
    grid = torch.linspace(1, 0, T+1)
    for i in range(T):
        x_next = mdlm_step(model, x, grid[i], grid[i+1], mask_id)
        assert (x_next == mask_id).sum() <= (x == mask_id).sum()
        deja_revele = (x != mask_id)
        assert torch.equal(x_next[deja_revele], x[deja_revele])
        x = x_next
    return x

In [5]:
import torch, transformers

tok = transformers.AutoTokenizer.from_pretrained("gpt2")
model = transformers.AutoModelForMaskedLM.from_pretrained(
    "/home/onyxia/work/ddpm/mdlm_hf", trust_remote_code=True).cuda().eval()

B, L, T = 1, 128, 128
mask_id = model.config.vocab_size - 1

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [6]:
x = mdlm_sample(model, B, L, T, mask_id)
print((x == mask_id).sum().item())   # doit valoir 0
print(tok.decode(x[0]))

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/home/onyxia/.cache/huggingface/modules/transformers_modules/mdlm_hf/4b10bc0bbcff5e70/modeling_mdlm_2.py:409: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=self.precision):
/home/onyxia/.cache/huggingface/modules/transformers_modules/mdlm_hf/4b10bc0bbcff5e70/modeling_mdlm_2.py:137: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/home/onyxia/.cache/huggingface/modules/transformers_modules/mdlm_hf/4b10bc0bbcff5e70/modeling_mdlm_2.py:301: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


0
<|endoftext|> as possible.

Advertisement

Would it easier to play the backup to the rest of “the gang” (U.S., Dempsey, etc.). Operating offense at a standard level (a 2-1-0). Or train with the offensive powers that would make the post-G.G.G. dynasty?

And how, if ultimately clinched, should each team have its own personal pitching staff — typically managed by W, Crews, etc.? Who would assign the responsibility for making the pitchers’ voices sound to take place without going away?<|endoftext|>Russian cyberweapon has allegedly played a role
